<a href="https://colab.research.google.com/github/psehgal2/Pandemaniac/blob/main/Strategy1_2_3_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
The MIT License (MIT)

Copyright (c) 2013-2014 California Institute of Technology

Permission is hereby granted, free of charge, to any person obtaining a copy of
this software and associated documentation files (the "Software"), to deal in
the Software without restriction, including without limitation the rights to
use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of
the Software, and to permit persons to whom the Software is furnished to do so,
subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS
FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR
COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER
IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN
CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
'''

__author__ = "Angela Gong (anjoola@anjoola.com)"

USAGE = '''
===========
   USAGE
===========

>>> import sim
>>> sim.run([graph], [dict with keys as names and values as a list of nodes])

Returns a dictionary containing the names and the number of nodes they got.

Example:
>>> graph = {"2": ["6", "3", "7", "2"], "3": ["2", "7, "12"], ... }
>>> nodes = {"strategy1": ["1", "5"], "strategy2": ["5", "23"], ... }
>>> sim.run(graph, nodes)
>>> {"strategy1": 243, "strategy6": 121, "strategy2": 13}

Possible Errors:
- KeyError: Will occur if any seed nodes are invalid (i.e. do not exist on the
            graph).
'''

from collections import Counter, OrderedDict
from copy import deepcopy
from random import randint


def run(adj_list, node_mappings):
  """
  Function: run
  -------------
  Runs the simulation on a graph with the given node mappings.

  adj_list: A dictionary representation of the graph adjacencies.
  node_mappings: A dictionary where the key is a name and the value is a list
                 of seed nodes associated with that name.
  """
  results = run_simulation(adj_list, node_mappings)
  return results

def run_simulations_2(adj_list, node_mappings):
    """
    Function: run_simulation
    ------------------------
    Runs the simulation. Returns a tuple with the overall result and a dictionary
    with the key as the "color"/name, and the value as the number of nodes that
    "color"/name got.

    adj_list: A dictionary representation of the graph adjacencies.
    node_mappings: A dictionary where the key is a name and the value is a list
                   of seed nodes associated with that name.
    """
    # Stores a mapping of nodes to their color.
    node_color = dict([(node, None) for node in adj_list.keys()])
    init(node_mappings, node_color)
    generation = 1

    # Keep calculating the epidemic until it stops changing. Randomly choose
    # number between 100 and 200 as the stopping point if the epidemic does not
    # converge.
    prev = None
    nodes = adj_list.keys()
    max_rounds = randint(100, 200)

    # Track the number of nodes conquered by each node
    node_conquer_count = dict([(node, 0) for node in adj_list.keys()])

    while not is_stable(generation, max_rounds, prev, node_color):
        prev = deepcopy(node_color)
        for node in nodes:
            (changed, color) = update(adj_list, prev, node)
            # Store the node's new color only if it changed.
            if changed:
                node_color[node] = color
                # Increment the conquer count for the node
                node_conquer_count[node] += 1

        # NOTE: prev contains the state of the graph of the previous generation,
        # node_colors contains the state of the graph at the current generation.
        # You could check these two dicts if you want to see the intermediate steps
        # of the epidemic.
        generation += 1

    # Return both the overall result and individual node contributions
    return get_result(node_mappings.keys(), node_color), node_conquer_count


def run_simulation(adj_list, node_mappings):
  """
  Function: run_simulation
  ------------------------
  Runs the simulation. Returns a dictionary with the key as the "color"/name,
  and the value as the number of nodes that "color"/name got.

  adj_list: A dictionary representation of the graph adjacencies.
  node_mappings: A dictionary where the key is a name and the value is a list
                 of seed nodes associated with that name.
  """
  # Stores a mapping of nodes to their color.
  node_color = dict([(node, None) for node in adj_list.keys()])
  init(node_mappings, node_color)
  generation = 1

  # Keep calculating the epidemic until it stops changing. Randomly choose
  # number between 100 and 200 as the stopping point if the epidemic does not
  # converge.
  prev = None
  nodes = adj_list.keys()
  max_rounds = randint(100, 200)
  while not is_stable(generation, max_rounds, prev, node_color):
    prev = deepcopy(node_color)
    for node in nodes:
      (changed, color) = update(adj_list, prev, node)
      # Store the node's new color only if it changSed.
      if changed: node_color[node] = color
    # NOTE: prev contains the state of the graph of the previous generation,
    # node_colros contains the state of the graph at the current generation.
    # You could check these two dicts if you want to see the intermediate steps
    # of the epidemic.
    generation += 1

  return get_result(node_mappings.keys(), node_color)


def init(color_nodes, node_color):
  """
  Function: init
  --------------
  Initializes the node to color mappings.
  """
  for (color, nodes) in color_nodes.items():
    for node in nodes:
      if node_color[node] is not None:
        node_color[node] = "__CONFLICT__"
      else:
        node_color[node] = color
  for (node, color) in node_color.items():
    if color == "__CONFLICT__":
      node_color[node] = None


def update(adj_list, node_color, node):
  """
  Function: update
  ----------------
  Updates each node based on its neighbors.
  """
  neighbors = adj_list[node]
  colored_neighbors = list(filter(None, [node_color[x] for x in neighbors]))
  total_votes = len(colored_neighbors)
  team_count = Counter(colored_neighbors)
  if node_color[node] is not None:
    team_count[node_color[node]] += 1.5
    total_votes += 1.5
  most_common = team_count.most_common(1)
  if len(most_common) > 0 and \
    most_common[0][1] > total_votes / 2.0:
    return (True, most_common[0][0])

  return (False, node_color[node])


def is_stable(generation, max_rounds, prev, curr):
  """
  Function: is_stable
  -------------------
  Checks whether or not the epidemic has stabilized.
  """
  if generation <= 1 or prev is None:
    return False
  if generation == max_rounds:
    return True
  for node, color in curr.items():
    if not prev[node] == curr[node]:
      return False
  return True


def get_result(colors, node_color):
  """
  Function: get_result
  --------------------
  Get the resulting mapping of colors to the number of nodes of that color.
  """
  color_nodes = {}
  for color in colors:
    color_nodes[color] = 0
  for node, color in node_color.items():
    if color is not None:
      color_nodes[color] += 1
  return color_nodes


if __name__ == '__main__':
  print(USAGE)



   USAGE

>>> import sim
>>> sim.run([graph], [dict with keys as names and values as a list of nodes])

Returns a dictionary containing the names and the number of nodes they got.

Example:
>>> graph = {"2": ["6", "3", "7", "2"], "3": ["2", "7, "12"], ... }
>>> nodes = {"strategy1": ["1", "5"], "strategy2": ["5", "23"], ... }
>>> sim.run(graph, nodes)
>>> {"strategy1": 243, "strategy6": 121, "strategy2": 13}

Possible Errors:
- KeyError: Will occur if any seed nodes are invalid (i.e. do not exist on the
            graph).



In [ ]:
class GraphProcessing:
    def __init__(self, filepath):
        self.filepath = filepath
        self.graph = None
        self.format = None
        self.num_seeds = None
        self.unique_id = None
        self.adjacency = None

    def get_graph(self):
        return self.graph

    def get_format(self):
        return self.format

    def get_num_seeds(self):
        return self.num_seeds

    def get_unique_id(self):
        return self.unique_id

    def get_adjacency(self):
      return self.adjacency

    def open_sampling_file(self):
        components = self.filepath.split('.')
        competition_format = components[0]
        num_seeds = int(components[1])
        unique_id = int(components[2])

        with open(file_path, "r") as file:
            file_contents = json.load(file)
        return file_contents, competition_format, num_seeds, unique_id

    def convert_to_graph(self):
        adjacency, competition_format, num_seeds, unique_id = self.open_sampling_file()
        self.adjacency = adjacency
        G = nx.Graph(adjacency)
        assert len(adjacency) == nx.number_of_nodes(G)
        self.graph = G
        self.format = competition_format
        self.num_seeds = num_seeds
        self.unique_id = unique_id

In [ ]:

import networkx as nx
import numpy as np
import json
import collections


class ThomsponSampling:
    def __init__(self, G, format, seeds, id, adj_list):
        self.graph = G
        self.iter = 0
        self.format = format
        self.num_seeds = seeds
        self.unique_id = id
        self.num_nodes = G.number_of_nodes()
        self.times = np.zeros(self.num_nodes)
        self.means = np.array([tup[1] for tup in G.degree], dtype=float)
        self.means /= np.max(self.means) + 1
        self.betweenness = nx.betweenness_centrality(self.graph, normalized=True)
        self.pagerank = nx.pagerank(self.graph)
        self.degree_centrality = nx.degree_centrality(self.graph)
        self.closeness_centrality = nx.closeness_centrality(self.graph)
        self.eigenvector_centrality = nx.eigenvector_centrality(self.graph)
        self.opp_strat = SimpleStrategies(G, seeds)
        self.adj_list = adj_list


    def get_score(self):
        return self.score

    def get_graph(self):
        return self.graph

    def get_iter(self):
        return self.iter

    def adjacency_matrix_to_dict(self):
      adjacency_dict = {}
      for i, row in enumerate(self.adj_list):
          neighbors = [str(j) for j, value in enumerate(row) if value == 1]
          adjacency_dict[str(i)] = neighbors
      return adjacency_dict

    def calculate_total_neighbors(self, start_node, max_level):
        visited = set()
        queue = collections.deque()
        queue.append(start_node)
        curr_level = 0
        level_count = []
        while queue and curr_level <= max_level:
            curr_level += 1
            level_len = len(queue)
            if curr_level != 1:
              level_count.append(level_len)
            for i in range(level_len):
                new_node = queue.popleft()
                neighbors = self.graph.neighbors(new_node)
                for neighbor in neighbors:
                    if neighbor not in visited:
                        queue.append(neighbor)
                        visited.add(neighbor)
        while len(level_count) < max_level:
          level_count.append(0)
        return level_count

    def get_overall_clustering(self, curr_node):
        transitivity = self.degree_centrality[curr_node]
        betweenness_centrality = self.betweenness[curr_node]
        closeness_centralities = self.closeness_centrality[curr_node]
        eigenvector_centralities = self.eigenvector_centrality[curr_node]
        deg_cent = self.degree_centrality[curr_node]
        return (1/2)*eigenvector_centralities + (1/2)*deg_cent


    def calculate_score_final(self, curr_node):
        # We want to assign values for level importance based on how clustered the graph is
        # If the graph is very clustered, then we want level importance = level_importance = [.4, .4, .2]
        # If the graph is not very clustered, we want level_importance = [.7, .2, .1]
        # We want to calculate clustering based on the combined clustering of the graph.

        # print(max(self.betweenness.values()) - min(self.betweenness.values()))
        # if max(self.betweenness.values()) - min(self.betweenness.values()) < 0.06:
        #   level_importance = [0.9, 0.05, 0.05]
        #   # level_importance = [0.99,0.05,0.05]
        #   # Here thompson sampling does not work
        # elif max(self.betweenness.values()) - min(self.betweenness.values()) < 0.1:
        #   level_importance = [0.5, 0.3, 0.2]
        # elif max(self.betweenness.values()) - min(self.betweenness.values()) < 0.15:
        #   level_importance = [0.4, 0.4, 0.2]
        # else:
        #   level_importance = [.7, .2, .1]

        # level_importance = [.5, .4, .1]
        level_importance = [.6, .25, .15]
        # level_importance = [.7, .2, .1]
        # level_importance = [.8, .15, .05]
        # level_importance = [0.4, 0.4, 0.2]
        # level_importance = [0.1, 0.2, 0.7]
        # level_importance = [.2, .2, .6]
        # level_importance = [.5, .3, .2]
        MAX_LEVEL = 3
        level_count = self.calculate_total_neighbors(curr_node, MAX_LEVEL)
        # assert len(level_importance) == len(level_count)
        score = 0
        for i in range(MAX_LEVEL):
          score += level_importance[i]*level_count[i]
        max_score = max(level_importance)*self.num_nodes
        score /= max_score
        betweenneess = self.betweenness.get(curr_node, 0)
        degree_centrality = self.degree_centrality.get(curr_node, 0)
        pagerank = self.pagerank.get(curr_node, 0)
        # return .6*score + .4*betweenneess
        return score

    def calculate_scores(self, nodes_list):
        score_list = []
        for curr_node in nodes_list:
          score_list.append(self.calculate_score_final(curr_node))
        return score_list

    def calculate_scores_2(self, nodes_list):
        score_list = []
        for curr_node in nodes_list:
          score_list.append(self.get_overall_clustering(curr_node))
        return score_list

    def thompsons_sampling(self):
        # k is num successes = num times a node is picked
        # beta distribution: alpha = alpha + k, beta = beta + (num trials - k)
        alpha_0 = 1
        beta_0 = 1
        self.iter = 0
        self.means = np.array([tup[1] for tup in self.graph.degree], dtype=float)
        self.means /= np.max(self.means) + 1
        self.times = np.zeros(self.num_nodes)
        sample_values = np.zeros(self.num_nodes)
        for i in range(self.num_nodes):
            alpha = alpha_0 + self.means[i]
            beta = beta_0 + (self.times[i] - self.means[i])
            sample_values = np.random.beta(alpha, beta)
            nodes = np.argsort(sample_values)[-self.num_seeds:][::-1]
            nodes_str = map(str, nodes)
            scores = self.calculate_scores_2(nodes_str)
            self.times[nodes] += 1
            # self.means[nodes] = self.means[nodes] + np.array(scores)/self.num_nodes
            self.means[nodes] += (np.array(scores) - self.means[nodes])/self.times[nodes]
        nodes = np.argsort(self.means)[-self.num_seeds:][::-1]
        return nodes

    def thompsons_sampling_new(self):
        #times is a list of the number of times each node has been picked
        # means is a list of estimated mean rewards for each node
        mu_bar = np.zeros(self.num_nodes)
        for i in range(self.num_nodes):
            self.iter += 1
            # mu_bar[i] = self.means[i] + np.sqrt(2 * np.log(self.iter) / self.times[i])
            mu_bar[i] = np.maximum(self.means[i] + np.sqrt(3 * np.log(self.iter) / (2 * self.times[i])), 2)
            nodes = np.argsort(mu_bar)[-self.num_seeds:][::-1]
            nodes_str = map(str, nodes)
            scores = self.calculate_scores_2(nodes_str)
            self.times[nodes] += 1
            self.means[nodes] += (np.array(scores) - self.means[nodes])/self.times[nodes]
        nodes = np.argsort(self.means)[-self.num_seeds:][::-1]
        return nodes

    def thompsons_sampling_v2(self):
        #times is a list of the number of times each node has been picked
        # means is a list of estimated mean rewards for each node
        self.iter = 0
        mu_bar = np.zeros(self.num_nodes)
        for i in range(self.num_nodes):
            self.iter += 1
            # mu_bar[i] = self.means[i] + np.sqrt(2 * np.log(self.iter) / self.times[i])
            mu_bar[i] = np.maximum(self.means[i] + np.sqrt(5 * np.log(self.iter) / (4 * self.times[i])), 2)
            nodes = np.argsort(mu_bar)[-self.num_seeds:][::-1]
            nodes_str = map(str, nodes)
            opp_nodes = self.opp_strat.get_highest_degree_nodes()
            opp_nodes = map(str, opp_nodes)
            opp_nodes = {"strategy1": nodes_str, "strategy2": opp_nodes}
            results = run(self.adj_list, opp_nodes)
            score = results['strategy1']
            self.times[nodes] += 1
            self.means[nodes] += score
        nodes = np.argsort(mu_bar)[-self.num_seeds:][::-1]
        return nodes


    def create_file(self, list_nums, filename):
        final_nums = []
        for i in range(len(list_nums)):
            final_nums.append(str(list_nums[i]))
        list_nums = final_nums
        list_nums = 50*list_nums
        with open(filename, "w") as wfile:
          for num in list_nums:
            wfile.write(str(num) + '\n')


In [ ]:
# SLAYYYYYYYYYYyYYYY
# strat to just find highest degree/highest clustering/highest betweenness nodes
class SimpleStrategies:
  def __init__(self, graph, k):
    self.graph = graph
    self.k = k

  def get_highest_degree_nodes(self):
    # gets k nodes with highest degrees in graph
    #k is integer
    # graph is networkx graph
    degrees = dict(self.graph.degree())
    # Sort nodes based on their degrees in descending order
    sorted_nodes = sorted(degrees, key=degrees.get, reverse=True)
    top_k_nodes = sorted_nodes[:self.k]

    return top_k_nodes

  def get_highest_clustering_nodes(self):
    # gets k nodes with highest clustering
    # k is an integer: how many nodes we want
    # graph is a networkx graph
    clustering = dict(nx.clustering(self.graph))
    sorted_nodes = sorted(clustering, key=clustering.get, reverse=True)
    top_k_nodes = sorted_nodes[:self.k]

    return top_k_nodes

  #We beat highest_betweenness_nodes only when our weights are low, but we lose to highest degree nodes then.
  # So we should account for this in our reward function.
  def get_highest_betweenness_nodes(self):
    #Consider nodes with high betweenness centrality.
    #Nodes with high betweenness centrality act as bridges between different parts of the network.
      betweenness_centrality = nx.betweenness_centrality(self.graph)
      return sorted(self.graph.nodes(), key=lambda x: betweenness_centrality[x], reverse=True)[:self.k]

  # We always beat connector nodes
  def get_connector_nodes(self):
    #Use community detection algorithms to identify clusters or groups of nodes within the network.
    # Target nodes that act as connectors between different clusters.
      communities = list(nx.algorithms.community.greedy_modularity_communities(self.graph))
      connector_nodes = []

      for community in communities:
          subgraph = self.graph.subgraph(community)
          edge_betweenness = nx.edge_betweenness_centrality(subgraph)
          max_edge = max(edge_betweenness, key=edge_betweenness.get)
          connector_nodes.extend(max_edge)

      # Deduplicate and take the top k connector nodes
      connector_nodes = list(set(connector_nodes))[:self.k]

      return connector_nodes

  # We always beat combined centrality
  def combined_centrality(self):
    # returns nodes with the combined centrality
    # each centrality is given a specific weight
    degree_weight = 0
    # closeness_weight = 0.3
    closeness_weight = 0.5
    betweenness_weight = 0.8
    eigenvector_weight = 0.5
    # clustering_weight = 0.1
    clustering_weight = 0.0
    # triangles_weight = 0.1
    triangles_weight = 0.0
    pagerank_weight = 0.8

    # Normalize the weights to ensure their sum is 1
    total_weight = sum([degree_weight, closeness_weight, betweenness_weight, eigenvector_weight, clustering_weight, triangles_weight])
    degree_weight /= total_weight
    closeness_weight /= total_weight
    betweenness_weight /= total_weight
    eigenvector_weight /= total_weight
    clustering_weight /= total_weight
    triangles_weight /= total_weight
    pagerank_weight /= total_weight

    weights = [degree_weight, closeness_weight, betweenness_weight, eigenvector_weight, clustering_weight, triangles_weight, pagerank_weight]

    centrality_measures = {
        'degree': nx.degree_centrality(self.graph),
        'closeness': nx.closeness_centrality(self.graph),
        'betweenness': nx.betweenness_centrality(self.graph),
        'eigenvector': nx.eigenvector_centrality(self.graph),
        'clustering': nx.clustering(self.graph),
        'pagerank': nx.pagerank(self.graph),
        'triangles': nx.triangles(self.graph)
    }

    nodes_combined_centrality = {}

    for node in self.graph.nodes():
        combined_centrality_value = sum(centrality_measures[measure][node] * weights[i] for i, measure in enumerate(centrality_measures))
        nodes_combined_centrality[node] = combined_centrality_value

    # Select the top k nodes with the highest combined centrality values
    top_nodes = sorted(nodes_combined_centrality, key=nodes_combined_centrality.get, reverse=True)[:self.k]

    return top_nodes

In [ ]:
class ClusteringStrategy:
    def __init__(self, G, seeds):
        self.graph = G
        self.num_seeds = seeds
        self.num_nodes = G.number_of_nodes()
        self.centrality_measures = {
        'degree': nx.degree_centrality(self.graph),
        'closeness': nx.closeness_centrality(self.graph),
        'betweenness': nx.betweenness_centrality(self.graph),
        'eigenvector': nx.eigenvector_centrality(self.graph),
        'clustering': nx.clustering(self.graph),
        'pagerank': nx.pagerank(self.graph),
        'triangles': nx.triangles(self.graph)
        }

    def get_score(self, curr_node, ts):
        cent_dict = self.centrality_measures
        degree = cent_dict["degree"][curr_node]
        betweenness = cent_dict["betweenness"][curr_node]
        eigenvector = cent_dict["eigenvector"][curr_node]
        clustering = cent_dict["clustering"][curr_node]
        triangles = cent_dict["triangles"][curr_node]
        score = .6*degree + .4*betweenness + .0*eigenvector + 0*clustering + .0*triangles
        ts_score = ts.calculate_score_final(curr_node)
        print(ts_score)
        return score + ts_score

    def get_k_best(self, ts):
        score_dict = {}
        for node in self.graph.nodes():
          score_dict[node] = self.get_score(node, ts)
        return sorted(self.graph.nodes(), key=lambda x: score_dict[x], reverse=True)[:self.num_seeds]

In [ ]:
if __name__ == "__main__":
    # file_path = "J.5.1.json"
    # on J.5.1 the combined clustering outperforms thompson sampling (0.7, 0.2, 0.1) by a large margin
    # on J.5.1 the thompson sampling (0.4, 0.4, 0.2) outperforms combined clustering
    file_path = "RR.10.51.json"
    g = GraphProcessing(file_path)
    g.convert_to_graph()

    G = g.get_graph()
    print("Edges:", len(G.edges))
    print ("clustering of G is:", nx.transitivity(G))
    format = g.get_format()
    k = g.get_num_seeds()
    id = g.get_unique_id()
    adj_list = g.get_adjacency()

    ts = ThomsponSampling(G, format, k, id, adj_list)
    chosen_nodes_ts = ts.thompsons_sampling_new()
    chosen_nodes_str_ts = map(str, chosen_nodes_ts)
    print('Chosen nodes 1: ', chosen_nodes_ts)
    ts.create_file(chosen_nodes_ts, "2chosennodes.txt")

    ss = SimpleStrategies(G, k)
    chosen_nodes_2 = ss.get_highest_degree_nodes()
    # chosen_nodes_2 = ss.get_highest_betweenness_nodes()
    # chosen_nodes_2 = ts.thompsons_sampling()
    chosen_nodes_2_str = map(str, chosen_nodes_2)
    chosen_nodes_ta = ["23", "148", "163", "55", "145", "109", "22", "107", "139", "24"]
    adjacency = g.get_adjacency()
    # print(run(adjacency, nodes))

    cs = ClusteringStrategy(G, k)
    chosen_nodes_cs = cs.get_k_best(ts)
    chosen_nodes_str_cs = map(str, chosen_nodes_cs)
    nodes = {"strategy1": chosen_nodes_str_ts, "strategy2": chosen_nodes_ta}
    adjacency = g.get_adjacency()
    print("Run2: ", run(adjacency, nodes))
    print("Clustering nodes:", chosen_nodes_ts)

Edges: 5522
clustering of G is: 0.2899375546650937


<ipython-input-68-231fbd329ad2>:173: RuntimeWarning: invalid value encountered in double_scalars
  mu_bar[i] = np.maximum(self.means[i] + np.sqrt(3 * np.log(self.iter) / (2 * self.times[i])), 2)
<ipython-input-68-231fbd329ad2>:173: RuntimeWarning: divide by zero encountered in double_scalars
  mu_bar[i] = np.maximum(self.means[i] + np.sqrt(3 * np.log(self.iter) / (2 * self.times[i])), 2)


Chosen nodes 1:  [188 247 268 241 257 242 296 246 209 180]
0.3438333333333334
0.42200000000000004
0.504
0.43116666666666664
0.4275
0.30433333333333334
0.5148333333333333
0.3105
0.4205
0.2255
0.3275
0.39116666666666666
0.3523333333333333
0.3565
0.2806666666666666
0.27116666666666667
0.26616666666666666
0.2673333333333333
0.24233333333333335
0.4798333333333333
0.041166666666666664
0.32183333333333336
0.5135
0.5656666666666667
0.48083333333333333
0.4298333333333333
0.3913333333333333
0.41633333333333333
0.4653333333333333
0.3255
0.3745
0.41583333333333333
0.3738333333333333
0.3616666666666667
0.4078333333333333
0.45149999999999996
0.4033333333333333
0.41183333333333333
0.33766666666666667
0.19699999999999998
0.29483333333333334
0.4556666666666666
0.3175
0.30133333333333334
0.32949999999999996
0.36683333333333334
0.37283333333333335
0.32466666666666666
0.24483333333333335
0.34833333333333333
0.18449999999999997
0.38316666666666666
0.34883333333333333
0.409
0.44350000000000006
0.55449999999

Clustering: .4469 - "J.10.20.json"
Lost to thompson 150 - 241
Lost to highest degree: 11 - 380

Clustering: .226 - "J.10.30.json"
Lost to max degree 55 - 1937
Lost to thompson 775 - 1213

Clustering: .11 - J.5.10
Won to thompson 88 - 43
Won to max degree 89 - 40

Clustering: .025 - RR.10.30
Won to thompson 122 - 81
Won to max degree 108 - 89

Clustering: .69 - RR.10.40
Lost to thompson 7 - 360
Lost to max degree 10 - 357

Clustering: .35 - RR.10.50
Won to thompson 283 - 209
Won to max degree = 296 - 200

Clustering: .01 clustering - RR.5.10
Won to thompson 196 - 115
Tie to max degree

Clustering: .08 clustering - R.5.20
Tie to thompson
Tie to max degree

Clustering: .4469 - "J.10.20.json"
Lost to thompson 150 - 241
Lost to highest degree: 11 - 380

Clustering: .226 - "J.10.30.json"
Lost to max degree 55 - 1937
Lost to thompson 775 - 1213

Clustering: .11 - J.5.10
Won to thompson 88 - 43
Won to max degree 89 - 40

Clustering: .025 - RR.10.30
Won to thompson 122 - 81
Won to max degree 108 - 89

Clustering: .69 - RR.10.40
Lost to thompson 7 - 360
Lost to max degree 10 - 357

Clustering: .35 - RR.10.50
Won to thompson 283 - 209
Won to max degree = 296 - 200

Clustering: .01 clustering - RR.5.10
Won to thompson 196 - 115
Tie to max degree

Clustering: .08 clustering - R.5.20
Tie to thompson
Tie to max degree

results:

For RR graph: clustering of G is: 0.3775438596491228
overall clustering of G is: 0.2089431847765763
- thompson sampling new w calculate score w betweenness vs max degree
  - level importance = [.7, .2, .1]: TS wins
  - level importance = [.6, .25, .15]: TS wins
  - level importance = [.5, .3, .2]: TS loses 41 - 58
  - level importance = [.4, .4, .2]: TS loses 25 - 75

For J graph: clustering of G is: 0.3129164531009739
overall clustering of G is: 0.1830372391818554
- thompson sampling new w calculate score w betweenness vs max degree
  - level importance = [.7, .2, .1]: 0-0 (picked same nodes)
  - level importance = [.6, .25, .15]: 0-0 (picked same nodes)
  - level importance = [.5, .3, .2]: 0-0 (picked same nodes)
  - level importance = [.4, .4, .2]: 130 - 64 TS wins

In [ ]:
def adjacency_matrix_to_dict(adjacency_matrix):
    adjacency_dict = {}
    for i, row in enumerate(adjacency_matrix):
        neighbors = [str(j) for j, value in enumerate(row) if value == 1]
        adjacency_dict[str(i)] = neighbors
    return adjacency_dict
# new test graph
# Set the number of nodes
num_nodes = 100

# Set the probability parameter
p = 0.08

# Create an Erdos-Renyi graph
er_graph = nx.erdos_renyi_graph(num_nodes, p)
k = 5
format = 'RR'
k = 5
id = 123
adjacency = nx.to_numpy_array(er_graph)
adjacency_dict = adjacency_matrix_to_dict(adjacency)
print(adjacency_dict)
er_graph = nx.Graph(adjacency_dict)
print("Edges:", len(er_graph.edges))

ts = ThomsponSampling(er_graph, format, k, id)
print ("overall clustering of G is:", ts.get_overall_clustering())
print("betweenness for G is: ", nx.betweenness_centrality(er_graph))
chosen_nodes = ts.thompsons_sampling_new()
chosen_nodes_str = map(str, chosen_nodes)
print(chosen_nodes)

ss = SimpleStrategies(er_graph, k)
chosen_nodes_2 = ss.get_highest_degree_nodes()
# chosen_nodes_2 = ss.get_highest_betweenness_nodes()
print('Chosen nodes 2: ', chosen_nodes_2)
# chosen_nodes_2 = ts.thompsons_sampling()
chosen_nodes_2_str = map(str, chosen_nodes_2)
nodes = {"strategy1": chosen_nodes_str, "strategy2": chosen_nodes_2_str}
print(run(adjacency_dict, nodes))

{'0': ['2', '9', '35', '38', '40', '42', '47', '60', '62', '77', '82', '84', '90', '93', '94'], '1': ['14', '19', '25', '34', '47', '48', '52', '53', '58', '61', '70', '73', '75', '76', '98'], '2': ['0', '8', '18', '35', '50', '56', '62', '90'], '3': ['9', '27', '51', '70', '75', '80', '93', '96', '97'], '4': ['7', '50', '61', '74', '94', '95'], '5': ['30', '31', '34', '47', '60', '67', '69', '77', '90'], '6': ['9', '49', '52', '54', '86'], '7': ['4', '20', '35', '72', '88'], '8': ['2', '16', '23', '32', '41', '51', '55', '68', '87', '89', '93', '97'], '9': ['0', '3', '6', '16', '31', '51', '59', '81', '86', '91'], '10': ['27', '40', '48', '50', '54', '67', '80', '87', '91', '97'], '11': ['20', '24', '27', '42', '45', '63', '75', '76'], '12': ['27', '35', '36', '52', '74', '81', '89'], '13': ['21', '29', '49', '56', '61', '65', '66', '68', '70', '79', '86'], '14': ['1', '18', '32', '50', '73', '78', '97'], '15': ['23', '39', '50', '75', '82'], '16': ['8', '9', '34', '40', '55', '58', '

<ipython-input-228-c8bcb5878530>:158: RuntimeWarning: invalid value encountered in double_scalars
  mu_bar[i] = np.maximum(self.means[i] + np.sqrt(7 * np.log(self.iter) / (6 * self.times[i])), 2)
<ipython-input-228-c8bcb5878530>:158: RuntimeWarning: divide by zero encountered in double_scalars
  mu_bar[i] = np.maximum(self.means[i] + np.sqrt(7 * np.log(self.iter) / (6 * self.times[i])), 2)


[7, 43, 48]
0.05401533923238985
[6, 46, 47]
0.05401533923238985
[15, 61, 24]
0.05401533923238985
[5, 35, 52]
0.05401533923238985
[7, 43, 48]
0.05401533923238985
[6, 46, 47]
0.05401533923238985
[10, 54, 35]
0.05401533923238985
[15, 61, 24]
0.05401533923238985
[8, 57, 35]
0.05401533923238985
[7, 43, 48]
0.05401533923238985
[6, 46, 47]
0.05401533923238985
[10, 54, 35]
0.05401533923238985
[15, 61, 24]
0.05401533923238985
[12, 59, 28]
0.05401533923238985
[6, 46, 47]
0.05401533923238985
[10, 54, 35]
0.05401533923238985
[7, 46, 45]
0.05401533923238985
[15, 61, 24]
0.05401533923238985
[10, 52, 38]
0.05401533923238985
[6, 46, 47]
0.05401533923238985
[10, 54, 35]
0.05401533923238985
[7, 46, 45]
0.05401533923238985
[15, 61, 24]
0.05401533923238985
[10, 47, 43]
0.05401533923238985
[10, 54, 35]
0.05401533923238985
[7, 46, 45]
0.05401533923238985
[12, 56, 32]
0.05401533923238985
[15, 61, 24]
0.05401533923238985
[7, 43, 48]
0.05401533923238985
[10, 54, 35]
0.05401533923238985
[7, 46, 45]
0.0540153392

Alternate strategies - want to implement something where we pick num_seeds + 5 nodes based on a mix of centrality and total degree (we want nodes with higher degree and centrality to contribute more to the overall dictionary score), and then we pick num_seeds from num_seeds + 5 nodes. for every combination of nodes, we run it against max degree and see how many nodes the combination claims. I have written another run simulation function that also returns the number of nodes each node claimed against the max degree simulation. We order then order each node based on how many nodes it was able to conquer against max degree, and we choose the top num_seeds from the ordered list.

In [ ]:
import math



def adjacency_matrix_to_dict(adjacency_matrix):
    adjacency_dict = {}
    for i, row in enumerate(adjacency_matrix):
        neighbors = [str(j) for j, value in enumerate(row) if value == 1]
        adjacency_dict[str(i)] = neighbors
    return adjacency_dict


def calculate_centralities(G):
    degree_centrality = nx.degree_centrality(G)
    eigenvector_centrality = nx.eigenvector_centrality(G)
    closeness_centrality = nx.closeness_centrality(G)
    betweenness_centrality = nx.betweenness_centrality(G)
    return degree_centrality, eigenvector_centrality, closeness_centrality, betweenness_centrality


def ordered_nodes(G, num_seeds):
  cs = ClusteringStrategy(G, num_seeds)

  centralities = calculate_centralities(G)
  d = {node: 0 for node in G.nodes()}
  # Update dictionary values based on centrality rankings (40% of all centralities)
  for centrality_measure in centralities:
      ranking = sorted(G.nodes(), key=centrality_measure.get, reverse=True)[:num_seeds]
      for node in ranking:
          d[node] += 1.0 * (num_seeds - ranking.index(node))
  # Update dictionary values based on degree (60% of the highest degree)
  # degree_ranking = sorted(G.nodes(), key=lambda node: G.degree(node), reverse=True)[:num_seeds]
  # for node in degree_ranking:
  #     d[node] += 0.2 * (num_seeds - degree_ranking.index(node))

  ordered_nodes_list = []
  for _ in range(12):
      ordered_nodes_list.extend(sorted(d, key=d.get, reverse=True)[:num_seeds])

  return ordered_nodes_list

def ordered_nodes_2(G, format, num_seeds, id, adj_list):
    ts = ThomsponSampling(G, format, num_seeds, id, adj_list)

    centralities = calculate_centralities(G)
    d = {node: 0 for node in G.nodes()}

    for centrality_measure in centralities:
        ranking = sorted(G.nodes(), key=centrality_measure.get, reverse=True)[:num_seeds]
        for node in ranking:
            level_importance = [0.7, 0.2, 0.1]
            MAX_LEVEL = 3
            level_count = ts.calculate_total_neighbors(node, MAX_LEVEL)
            score = 0
            for i in range(MAX_LEVEL):
                score += level_importance[i] * level_count[i]

            betweenness = ts.betweenness.get(node, 0)
            degree_centrality = ts.degree_centrality.get(node, 0)
            pagerank = ts.pagerank.get(node, 0)
            d[node] += 0.25 * score + 0.25 * betweenness + 0.25 * degree_centrality + 0.25 * pagerank

    ordered_nodes_list = []
    for _ in range(12):
        ordered_nodes_list.extend(sorted(d, key=d.get, reverse=True)[:num_seeds])

    return ordered_nodes_list

def get_highest_degree_nodes(G, num_seeds):
  degrees = dict(G.degree())
  sorted_nodes = sorted(degrees, key=degrees.get, reverse=True)
  top_k_nodes = sorted_nodes[:num_seeds]
  return top_k_nodes

def get_best_nodes_learning(G, format, num_seeds, id, adj_list):
  ordered_nodes_list = ordered_nodes_2(G, format, num_seeds, id, adj_list)
  final_nodes = set()
  num_seeds_new = int(1.5*num_seeds)
  for i in range(math.comb(num_seeds_new, num_seeds)):
    # every possible combination of num_seeds in ordered_nodes_list
    # Our strat
    node_combination = ordered_nodes_list[i * num_seeds : (i + 1) * num_seeds]
    node_combination_str = map(str, node_combination)
    #opp strat
    max_deg = get_highest_degree_nodes(G, num_seeds)
    max_deg_str = map(str, max_deg)
    # Remember to change seed_combinations to be the format of a dictionary that they give in
    nodes = {"self": node_combination_str, "max_degree": max_deg_str}
    adjacency = nx.to_numpy_array(G)
    adjacency = adjacency_matrix_to_dict(adjacency)
    overall_result, node_contributions = run_simulations_2(adjacency, nodes)
    print(overall_result)
    print(node_contributions)
    ordered_seeds = sorted(node_combination, key=lambda node: node_contributions.get(node, 0), reverse=True)
    final_nodes.update(ordered_seeds[:num_seeds])
  return final_nodes


file_path = "J.5.11.json"
g = GraphProcessing(file_path)
g.convert_to_graph()

G = g.get_graph()
format = g.get_format()
k = g.get_num_seeds()
id = g.get_unique_id()
adj_list = g.get_adjacency()

ts = ThomsponSampling(G, format, k, id, adj_list)
chosen_nodes = list(get_best_nodes_learning(G, format, k, id, adj_list))
print(type(chosen_nodes))
chosen_nodes_str = map(str, chosen_nodes_2)
ts.create_file(chosen_nodes, "5chosennodes_ts.txt")
adjacency = g.get_adjacency()
print(run(adjacency, nodes))

ss = SimpleStrategies(G, k)
chosen_nodes_2 = ss.get_highest_degree_nodes()
chosen_nodes_2 = ss.get_highest_betweenness_nodes()
chosen_nodes_2_str = map(str, chosen_nodes_2)
nodes = {"strategy1": chosen_nodes_str, "strategy2": chosen_nodes_2_str}
adjacency = g.get_adjacency()
# ts.create_file(chosen_nodes_2, "6chosennodes_ts.txt")



<ipython-input-4-c9f176a6da8e>:166: RuntimeWarning: invalid value encountered in double_scalars
  mu_bar[i] = np.maximum(self.means[i] + np.sqrt(7 * np.log(self.iter) / (6 * self.times[i])), 2)
<ipython-input-4-c9f176a6da8e>:166: RuntimeWarning: divide by zero encountered in double_scalars
  mu_bar[i] = np.maximum(self.means[i] + np.sqrt(7 * np.log(self.iter) / (6 * self.times[i])), 2)


TypeError: ordered_nodes_2() missing 1 required positional argument: 'clustering_strategy'